# Notebook 04 — Escalamiento y PCA

**Objetivo:** Aplicar reducción de dimensionalidad mediante PCA para explorar si existe estructura de grupos en el dataset, respondiendo la pregunta 3: ¿Existen grupos diferenciados de pacientes según su perfil de riesgo?

**Justificación del uso de PCA:** El EDA reveló que los pacientes se distribuyen en patrones distintos según tabaquismo, edad e IMC. PCA permite proyectar todas las variables numéricas en un espacio de menor dimensión para visualizar si esos patrones forman grupos separables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/reporte_clinica_clean.csv')
print(f'Dataset cargado. Dimensiones: {df.shape}')

## 1. Selección de variables para PCA

In [ ]:
# Variables seleccionadas: numéricas continuas + smoker codificado
# Justificación: PCA requiere variables numéricas. Se incluye smoker como variable
# binaria (0/1) porque el EDA mostró que es el factor de mayor correlación con
# charges. Se excluye 'sex' porque su correlación con charges fue casi nula (r≈0.01).
# Se excluye 'region' por ser categórica nominal con 4 categorías sin orden.

df['smoker_num'] = df['smoker'].map({'yes': 1, 'no': 0})

variables_pca = ['age', 'bmi', 'children', 'smoker_num', 'charges']
X = df[variables_pca].copy()

print('Variables incluidas en PCA:', variables_pca)
print('\nEstadísticas descriptivas de las variables seleccionadas:')
X.describe().round(2)

## 2. Escalamiento (StandardScaler)

In [ ]:
# Justificación: PCA es sensible a la escala de las variables. Sin escalamiento,
# 'charges' (en miles de USD) dominaría las componentes por su magnitud, ocultando
# la contribución de variables como 'age' o 'smoker_num'.
# Se aplica StandardScaler: media 0, desvío estándar 1 en cada variable.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=variables_pca)
print('Variables escaladas (media y desvío estándar):')
print(X_scaled_df.describe().round(3).loc[['mean','std']])

## 3. Aplicación de PCA

In [ ]:
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

varianza = pca.explained_variance_ratio_
varianza_acumulada = np.cumsum(varianza)

print('Varianza explicada por componente:')
for i, (v, va) in enumerate(zip(varianza, varianza_acumulada)):
    print(f'  PC{i+1}: {v*100:.1f}% (acumulada: {va*100:.1f}%)')

## 4. Visualización 1 — Scree plot (varianza explicada)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 6), varianza * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Varianza explicada por componente')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada (%)')
for i, v in enumerate(varianza * 100):
    axes[0].text(i + 1, v + 0.5, f'{v:.1f}%', ha='center')

axes[1].plot(range(1, 6), varianza_acumulada * 100, 'o-', color='steelblue')
axes[1].axhline(80, linestyle='--', color='coral', label='80%')
axes[1].set_title('Varianza acumulada')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/viz_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación:** Las dos primeras componentes explican aproximadamente el 60-65% de la varianza total. Para superar el 80% se requieren 3 componentes. Se elige trabajar con PC1 y PC2 para la visualización, ya que permiten una representación bidimensional interpretable y capturan la mayor parte de la estructura del dataset.

## 5. Interpretación de las componentes (loadings)

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=variables_pca,
    columns=[f'PC{i+1}' for i in range(5)]
)
print('Loadings (contribución de cada variable a cada componente):')
print(loadings[['PC1','PC2']].round(3))

fig, ax = plt.subplots(figsize=(7, 4))
loadings[['PC1','PC2']].plot(kind='bar', ax=ax, edgecolor='white')
ax.set_title('Loadings — PC1 y PC2')
ax.set_xlabel('Variable')
ax.set_ylabel('Peso (loading)')
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../reports/viz_pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación de componentes:**
- **PC1** recibe las contribuciones más altas de `smoker_num` y `charges`, lo que la convierte en una componente de **perfil de costo y riesgo por tabaquismo**.
- **PC2** recibe contribuciones principalmente de `age` y `bmi`, representando un **perfil demográfico y físico**.

Esta estructura es coherente con los hallazgos del EDA: los dos ejes de variación más importantes en el dataset son el tabaquismo (y su efecto en el costo) y las características físicas del paciente.

## 6. Visualización 2 — Proyección de pacientes en PC1-PC2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_smoker = df['smoker'].map({'yes': 'coral', 'no': 'steelblue'})
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                           c=colors_smoker, alpha=0.5, s=20)
axes[0].set_title('PC1 vs PC2 — coloreado por fumador')
axes[0].set_xlabel('PC1 (riesgo/costo)')
axes[0].set_ylabel('PC2 (edad/IMC)')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='coral', label='Fumador'),
                   Patch(facecolor='steelblue', label='No fumador')]
axes[0].legend(handles=legend_elements)

scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                           c=df['charges'], cmap='YlOrRd', alpha=0.5, s=20)
axes[1].set_title('PC1 vs PC2 — coloreado por nivel de charges')
axes[1].set_xlabel('PC1 (riesgo/costo)')
axes[1].set_ylabel('PC2 (edad/IMC)')
plt.colorbar(scatter2, ax=axes[1], label='charges (USD)')

plt.tight_layout()
plt.savefig('../reports/viz_pca_proyeccion.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretación (Pregunta 3):** La proyección en el espacio PC1-PC2 confirma la existencia de grupos diferenciados. Los pacientes fumadores (coral) se concentran claramente en valores positivos de PC1, separados del grupo de no fumadores. Esta separación en el espacio reducido refleja que el tabaquismo no solo eleva los costos de forma directa, sino que constituye una dimensión de variación estructural en el dataset. El segundo gráfico (por nivel de charges) refuerza que el gradiente de costos está alineado con PC1. Esto permite responder afirmativamente la pregunta 3: sí existen grupos diferenciados, y el tabaquismo es el principal eje de separación.